In [2]:
import psi4
import pandas as pd
import os
import numpy as np
import os
import sys
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)
%load_ext autoreload
%autoreload 2
from src.lps_rscf import lps_solver

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
csv_file = '../data/sic_closed_shell_atoms.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"-> Loaded existing results: {len(df)} rows found.")
else:
    df = pd.DataFrame()
    print("-> No existing file found. Starting fresh.")

-> No existing file found. Starting fresh.


In [13]:
ATOMS = {
    'He':  {'mult': 1,'N': 2}, 
    'Be': {'mult': 1,'N': 4},
    'Ne': {'mult': 1,'N': 10}, 
    'Mg': {'mult': 1,'N': 12},
    'Ar':  {'mult': 1,'N': 18}, 
    'Ca':  {'mult': 1,'N': 20},
    'Zn':  {'mult': 1,'N': 30}, 
    'Kr':  {'mult': 1,'N': 36}
}
psi4.set_options({'basis': 'UGBS_S', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

METHOD = 'W FA'
tf_scaling = 0.0
MAX_ITER = 1000
DAMPING = [0.9, 0.0, 0.001]
DIIS_OPT = [True, True]

for atom in ATOMS:
    if not df.empty:
        exists = df[
            (df['Atom'] == atom) & 
            (df['Method'] == METHOD) & 
            (df['Basis'] == psi4.core.get_global_option("BASIS"))
        ]
        if not exists.empty:
            print(f"Skipping {atom} (Already exists for {METHOD}/{psi4.core.get_global_option("BASIS")})")
            continue

    print(f"Calculating {atom} with {METHOD}...")
    mol = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    try:
        E, D, mu, iterations, e_list, e_conv, d_conv = lps_solver(
            mol=mol,
            E_conv=1.0e-5,
            D_conv=1.0e-5,
            maxiter=MAX_ITER,
            TP=['LDA_K_TF', tf_scaling],
            lam=1.0,
            EXC=['LDA_X', 0.0, 'LDA_C_VWN', 0.0],
            FA=[True, 1.0],
            damp=DAMPING,
            DIIS=DIIS_OPT,
            Guess=None,
            lehtomaki=False,
            verbose=False
        )
        if iterations >= MAX_ITER:
            print("  !!! SCF failed to converge (Max cycles exceeded).")
        else:
            print('\nFinal SCF energy: %.4f Hartree' \
                '\nConverged in %.i iterations' % ( E, iterations))
            row = {
                "Method": METHOD,
                "Atom": atom,
                "Basis": psi4.core.get_global_option("BASIS"),
                "Grid_Sph": psi4.core.get_global_option("DFT_SPHERICAL_POINTS"),
                "Grid_Rad": psi4.core.get_global_option("DFT_RADIAL_POINTS"),
                "Energy,Ha": round(E, 6),
                "ChemPot,Ha": round(mu, 6),
                "Iterations": iterations,
                "DIIS": DIIS_OPT,
                "Damp_Start": DAMPING[0],
                "Damp_End": DAMPING[1],
                "Damp_Cutoff": DAMPING[2]
            }
            
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)

    except Exception as e:
        print(f"  !!! Failed {atom}. Error: {e}")
        continue

Calculating He with W FA...

Final SCF energy: -2.8617 Hartree
Converged in 39 iterations
Calculating Be with W FA...

Final SCF energy: -19.0194 Hartree
Converged in 64 iterations
Calculating Ne with W FA...

Final SCF energy: -264.3516 Hartree
Converged in 83 iterations
Calculating Mg with W FA...

Final SCF energy: -450.7824 Hartree
Converged in 88 iterations
Calculating Ar with W FA...

Final SCF energy: -1487.9793 Hartree
Converged in 100 iterations
Calculating Ca with W FA...

Final SCF energy: -2032.0474 Hartree
Converged in 104 iterations
Calculating Zn with W FA...


/Users/ivanbosko/Documents/CODES/GIT/DFT-Tutorials/LPS/src/lps_rscf.py:157: RuntimeWarning: divide by zero encountered in log10
  e_conv_list.append(np.log10(abs(SCF_E - Eold)))



Final SCF energy: -6766.8701 Hartree
Converged in 119 iterations
Calculating Kr with W FA...

Final SCF energy: -11640.9208 Hartree
Converged in 124 iterations


In [14]:
df.to_csv(csv_file, index=False)
display(df)

,Method,Atom,Basis,Grid_Sph,Grid_Rad,"Energy,Ha","ChemPot,Ha",Iterations,DIIS,Damp_Start,Damp_End,Damp_Cutoff
0,W FA,He,UGBS_S,6,1000,-2.861680,-0.917956,39,"[True, True]",0.9,0.0,0.001
1,W FA,Be,UGBS_S,6,1000,-19.019442,-2.022702,64,"[True, True]",0.9,0.0,0.001
2,W FA,Ne,UGBS_S,6,1000,-264.351611,-7.541360,83,"[True, True]",0.9,0.0,0.001
3,W FA,Mg,UGBS_S,6,1000,-450.782450,-10.119237,88,"[True, True]",0.9,0.0,0.001
4,W FA,Ar,UGBS_S,6,1000,-1487.979296,-20.070485,100,"[True, True]",0.9,0.0,0.001
5,W FA,Ca,UGBS_S,6,1000,-2032.047374,-24.126936,104,"[True, True]",0.9,0.0,0.001
6,W FA,Zn,UGBS_S,6,1000,-6766.870137,-49.955267,119,"[True, True]",0.9,0.0,0.001
7,W FA,Kr,UGBS_S,6,1000,-11640.920803,-69.889360,124,"[True, True]",0.9,0.0,0.001


In [7]:
ATOMS = {
    'He':  {'mult': 1}, 
    'Be': {'mult': 1},
    'Ne': {'mult': 1}, 
    'Mg': {'mult': 1},
    'Ar':  {'mult': 1}, 
    'Ca':  {'mult': 1},
    'Zn':  {'mult': 1}, 
    'Kr':  {'mult': 1}
}
psi4.core.set_output_file('output.dat', False)
psi4.set_options({'basis': 'UGBS',
                  'scf_type': 'PK'})
rhf_energies = {}
rhf_homos = {}
for atom in ATOMS:
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    E, wfn = psi4.energy('SCF', return_wfn=True)
    homo = wfn.epsilon_a().np[wfn.nalpha()-1]
    rhf_energies[atom] = round(E, 6)
    rhf_homos[atom] = round(homo, 6)
    print(f"RHF/UGBS {atom} Energy: {E:.4f} Hartree, IP: {homo:.4f}")

RHF/UGBS He Energy: -2.8617 Hartree, IP: -0.9180
RHF/UGBS Be Energy: -14.5730 Hartree, IP: -0.3093
RHF/UGBS Ne Energy: -128.5471 Hartree, IP: -0.8504
RHF/UGBS Mg Energy: -199.6146 Hartree, IP: -0.2530
RHF/UGBS Ar Energy: -526.8175 Hartree, IP: -0.5910
RHF/UGBS Ca Energy: -676.7582 Hartree, IP: -0.1955
RHF/UGBS Zn Energy: -1777.8481 Hartree, IP: -0.2925
RHF/UGBS Kr Energy: -2752.0549 Hartree, IP: -0.5242


In [16]:
atom_order = ['He', 'Be', 'Ne', 'Mg', 'Ar', 'Ca', 'Zn', 'Kr']
energy_table = df.pivot(index='Atom', columns='Method', values='Energy,Ha')
energy_table = energy_table.reindex(atom_order)
energy_table['RHF/UGBS'] = pd.Series(rhf_energies)
new_order = ['W FA', 'RHF/UGBS']
energy_table = energy_table[new_order]

reference = energy_table['RHF/UGBS']

res = {}
for method, energies in energy_table.items():
    if method == 'RHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
energy_table = pd.concat([energy_table, stats_df], sort=False)
display(energy_table)

,W FA,RHF/UGBS
He,-2.861680,-2.861680
Be,-19.019442,-14.573023
Ne,-264.351611,-128.547083
Mg,-450.782450,-199.614621
Ar,-1487.979296,-526.817486
Ca,-2032.047374,-676.758154
Zn,-6766.870137,-1777.848060
Kr,-11640.920803,-2752.054860
MAE(Ha),2073.220000,NaN
rMAE(%),156.040000,NaN


In [17]:
mu_table = df.pivot(index='Atom', columns='Method', values='ChemPot,Ha')
mu_table = mu_table.reindex(atom_order)
mu_table['RHF/UGBS'] = pd.Series(rhf_homos)
mu_table = mu_table[new_order]

reference = mu_table['RHF/UGBS']

res = {}
for method, energies in mu_table.items():
    if method == 'RHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
mu_table = pd.concat([mu_table, stats_df], sort=False)

display(mu_table)

,W FA,RHF/UGBS
He,-0.917956,-0.917956
Be,-2.022702,-0.309271
Ne,-7.541360,-0.850411
Mg,-10.119237,-0.253048
Ar,-20.070485,-0.590989
Ca,-24.126936,-0.195527
Zn,-49.955267,-0.292463
Kr,-69.889360,-0.524161
MAE(Ha),22.590000,NaN
rMAE(%),6373.720000,NaN
